# 🎬 Hệ Thống Gợi Ý Phim Hybrid trên Nền Tảng Big Data
**Đồ án cuối khóa | Môn: Dữ liệu lớn (Big Data)**

---

Notebook này trình bày toàn bộ quy trình xây dựng hệ thống gợi ý phim kết hợp:
- **Collaborative Filtering (ALS):** Gợi ý dựa trên hành vi cộng đồng.
- **Content-Based Filtering (Genome):** Gợi ý dựa trên đặc điểm nội dung phim.
- **Hybrid Model:** Kết hợp cả hai để tăng độ chính xác.

**Dataset:** MovieLens 20M (~20 triệu lượt đánh giá, 27,000 bộ phim)

## 1. Cài đặt thư viện

In [ ]:
!pip install pyspark kagglehub matplotlib seaborn pandas numpy -q

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col, explode, avg

sns.set_theme(style='whitegrid', palette='pastel')
print('✅ Import thành công!')

## 2. Khởi tạo Spark Session
Chúng ta sử dụng **Apache Spark** để xử lý tập dữ liệu 20 triệu bản ghi — quá lớn để xử lý bằng Pandas thông thường.

In [ ]:
spark = SparkSession.builder \
    .appName("MovieLens_Hybrid_Recommendation") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print(f'✅ Spark {spark.version} khởi động thành công!')

## 3. Tải và Chuẩn bị Dữ liệu
### Cấu trúc dữ liệu MovieLens 20M:
| File | Mô tả | Kích thước |
|------|--------|------------|
| `rating.csv` | Lịch sử đánh giá của người dùng | ~20M dòng |
| `movie.csv` | Thông tin phim (tiêu đề, thể loại) | ~27K dòng |
| `genome_scores.csv` | "DNA" đặc điểm của từng bộ phim | ~11M dòng |

In [ ]:
print('--- Đang tải dữ liệu từ Kaggle... ---')
dataset_dir = kagglehub.dataset_download('grouplens/movielens-20m-dataset')

rating_path  = os.path.join(dataset_dir, 'rating.csv')
movie_path   = os.path.join(dataset_dir, 'movie.csv')
genome_path  = os.path.join(dataset_dir, 'genome_scores.csv')

ratings_df = spark.read.csv(rating_path,  header=True, inferSchema=True)
movies_df  = spark.read.csv(movie_path,   header=True, inferSchema=True)
genome_df  = spark.read.csv(genome_path,  header=True, inferSchema=True)

# Tiền xử lý
data = ratings_df.select(
    col('userId').cast('int'),
    col('movieId').cast('int'),
    col('rating').cast('float')
).dropna()

print(f'✅ ratings_df: {ratings_df.count():,} dòng')
print(f'✅ movies_df:  {movies_df.count():,} dòng')
data.show(5)

## 4. Phân Tích Khám Phá Dữ Liệu (EDA)
Chúng ta sẽ trực quan hóa dữ liệu để hiểu rõ hơn về tập dữ liệu trước khi đưa vào mô hình.

In [ ]:
# A. Phân phối điểm đánh giá (Rating Distribution)
# Lấy mẫu 1% để vẽ biểu đồ nhanh hơn
sample_ratings = data.sample(fraction=0.01, seed=42).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Biểu đồ cột
rating_counts = sample_ratings['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index.astype(str), rating_counts.values, color=sns.color_palette('viridis', len(rating_counts)))
axes[0].set_title('Phân phối điểm đánh giá', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Điểm Rating')
axes[0].set_ylabel('Số lượng (mẫu 1%)')

# Biểu đồ tròn cho rating phổ biến nhất
top_ratings = rating_counts.nlargest(5)
axes[1].pie(top_ratings.values, labels=top_ratings.index.astype(str), autopct='%1.1f%%', startangle=90, colors=sns.color_palette('viridis', 5))
axes[1].set_title('Top 5 điểm Rating phổ biến nhất', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'📊 Điểm trung bình: {sample_ratings["rating"].mean():.2f}')

In [ ]:
# B. Top 15 thể loại phim phổ biến nhất
movies_pd = movies_df.toPandas()
all_genres = movies_pd['genres'].str.split('|').explode()
genre_counts = all_genres[all_genres != '(no genres listed)'].value_counts().head(15)

plt.figure(figsize=(12, 7))
colors = sns.color_palette('coolwarm', len(genre_counts))
bars = plt.barh(genre_counts.index, genre_counts.values, color=colors)
plt.bar_label(bars, fmt='%d', padding=3)
plt.title('Top 15 Thể loại phim phổ biến nhất trong MovieLens 20M', fontsize=14, fontweight='bold')
plt.xlabel('Số lượng phim')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# C. Phân phối số lượng rating mỗi người dùng
ratings_per_user = data.groupBy('userId').count().sample(fraction=0.05, seed=42).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(ratings_per_user['count'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Phân phối số lượng phim mỗi user đã đánh giá', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Số phim đã đánh giá')
axes[0].set_ylabel('Số người dùng')

# D. Phân phối điểm Relevance của Genome
sample_genome = genome_df.sample(fraction=0.005, seed=42).toPandas()
axes[1].hist(sample_genome['relevance'], bins=40, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Phân phối điểm Relevance của Genome Tags', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Relevance Score (0 → 1)')
axes[1].set_ylabel('Tần suất')

plt.tight_layout()
plt.show()
print('💡 Phần lớn genome scores gần 0 (không liên quan) → chứng tỏ data rất thưa (sparse), phù hợp cho ALS.')

## 5. Xây dựng Mô hình Hybrid
### 5.1. Collaborative Filtering — Thuật toán ALS
ALS (Alternating Least Squares) phân rã ma trận User-Item thành 2 ma trận đặc trưng nhỏ hơn:
$$R \approx U \times V^T$$
Từ đó dự đoán được điểm rating cho các phim người dùng chưa xem.

In [ ]:
(training, test) = data.randomSplit([0.8, 0.2], seed=42)
print(f'Training: {training.count():,} | Test: {test.count():,}')

als = ALS(
    maxIter=10,
    regParam=0.05,
    rank=20,
    userCol='userId',
    itemCol='movieId',
    ratingCol='rating',
    coldStartStrategy='drop',
    nonnegative=True
)

print('--- Đang huấn luyện ALS... ---')
model = als.fit(training)

predictions = model.transform(test)
evaluator = RegressionEvaluator(metricName='rmse', labelCol='rating', predictionCol='prediction')
rmse = evaluator.evaluate(predictions)
print(f'✅ Huấn luyện hoàn tất | RMSE: {rmse:.4f}')

In [ ]:
# Visualize RMSE so sánh (ALS vs. Baseline)
baseline_rmse = data.agg(avg('rating')).collect()[0][0]
# Baseline: dự đoán tất cả bằng điểm trung bình
baseline = test.withColumn('prediction', col('rating') * 0 + baseline_rmse)
baseline_rmse_val = evaluator.evaluate(baseline)

fig, ax = plt.subplots(figsize=(8, 5))
models = ['Baseline\n(Trung bình)', 'ALS\n(Collaborative)']
rmse_vals = [baseline_rmse_val, rmse]
colors = ['#ff6b6b', '#51cf66']
bars = ax.bar(models, rmse_vals, color=colors, width=0.4, edgecolor='white')
ax.bar_label(bars, fmt='%.4f', padding=5, fontsize=12, fontweight='bold')
ax.set_title('So sánh RMSE: Baseline vs. ALS', fontsize=14, fontweight='bold')
ax.set_ylabel('RMSE (càng thấp càng tốt)')
ax.set_ylim(0, max(rmse_vals) * 1.3)
plt.tight_layout()
plt.show()
print(f'📉 ALS giảm RMSE {((baseline_rmse_val - rmse)/baseline_rmse_val*100):.1f}% so với Baseline!')

### 5.2. Content-Based Filtering — Genome Data
Mỗi bộ phim được biểu diễn bằng một vector đặc điểm từ `genome_scores.csv`. Ta tính độ tương đồng giữa phim user thích nhất và các phim còn lại.

In [ ]:
def get_similar_movies(movie_id, top_n=10):
    target_tags = genome_df.filter(
        (col('movieId') == movie_id) & (col('relevance') > 0.5)
    ).select('tagId').collect()
    tag_ids = [r.tagId for r in target_tags]

    if not tag_ids:
        return spark.createDataFrame([], 'movieId int, score float')

    return genome_df.filter(col('tagId').isin(tag_ids)) \
        .groupBy('movieId') \
        .agg(avg('relevance').alias('score')) \
        .filter(col('movieId') != movie_id) \
        .orderBy(col('score').desc()) \
        .limit(top_n)

print('✅ Hàm Content-Based Filtering đã sẵn sàng!')

### 5.3. Hybrid Model
$$Score_{final} = 0.7 \times Score_{ALS} + 0.3 \times Score_{CBF}$$
- **70% ALS**: Tận dụng trí tuệ cộng đồng.
- **30% CBF**: Đảm bảo tính liên quan về nội dung, xử lý Cold Start.

In [ ]:
def recommend_hybrid(user_id, top_n=10):
    user_df = spark.createDataFrame([(user_id,)], ['userId'])
    als_recs = model.recommendForUserSubset(user_df, top_n)
    flat_als = als_recs.withColumn('rec', explode('recommendations')) \
        .select(col('rec.movieId').alias('movieId'), col('rec.rating').alias('als_score'))

    top_movie = data.filter((col('userId') == user_id) & (col('rating') >= 4.0)) \
        .orderBy(col('rating').desc()).select('movieId').first()

    if top_movie:
        cb_recs = get_similar_movies(top_movie.movieId, top_n)
        final = flat_als.join(cb_recs, on='movieId', how='outer') \
            .fillna(0, subset=['als_score', 'score']) \
            .withColumn('hybrid_score', col('als_score') * 0.7 + col('score') * 0.3)
    else:
        final = flat_als.withColumn('hybrid_score', col('als_score'))

    return final.orderBy(col('hybrid_score').desc()).limit(top_n) \
        .join(movies_df, on='movieId') \
        .select('title', 'genres', 'hybrid_score') \
        .toPandas()

print('✅ Hàm Hybrid Recommendation đã sẵn sàng!')

## 6. Demo & Trực Quan Hóa Kết Quả

In [ ]:
# Thay đổi user_id để xem gợi ý cho người dùng khác
user_id = 1

print(f'\n--- Đang tạo gợi ý Hybrid cho User {user_id}... ---')
result = recommend_hybrid(user_id, top_n=10)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Biểu đồ thanh ngang: điểm Hybrid
colors = sns.color_palette('magma', len(result))
axes[0].barh(result['title'], result['hybrid_score'], color=colors)
axes[0].set_title(f'Top 10 Phim Gợi Ý cho User {user_id}\n(Hybrid Score)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hybrid Score')
axes[0].invert_yaxis()

# Biểu đồ tròn: phân bổ thể loại trong kết quả
genres_in_result = result['genres'].str.split('|').explode().value_counts().head(8)
axes[1].pie(genres_in_result.values, labels=genres_in_result.index, autopct='%1.0f%%',
            startangle=90, colors=sns.color_palette('Set2', len(genres_in_result)))
axes[1].set_title(f'Phân bổ thể loại trong kết quả gợi ý\n(User {user_id})', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n📋 Chi tiết kết quả:')
result

In [ ]:
# So sánh kết quả gợi ý cho nhiều người dùng
user_ids = [1, 2, 3]
fig, axes = plt.subplots(1, len(user_ids), figsize=(20, 7), sharey=False)

for i, uid in enumerate(user_ids):
    res = recommend_hybrid(uid, top_n=10)
    colors = sns.color_palette('viridis', len(res))
    axes[i].barh(res['title'], res['hybrid_score'], color=colors)
    axes[i].set_title(f'User {uid}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Hybrid Score')
    axes[i].invert_yaxis()

plt.suptitle('So sánh kết quả gợi ý Hybrid cho 3 người dùng khác nhau', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('💡 Mỗi người dùng nhận được danh sách phim khác nhau — đây là tính cá nhân hóa của hệ thống!')

## 7. Kết luận

| Tiêu chí | Kết quả |
|---|---|
| **Công cụ Big Data** | Apache PySpark xử lý 20M bản ghi |
| **RMSE của mô hình** | Thấp hơn đáng kể so với Baseline |
| **Giải quyết Cold Start** | Dùng Content-Based (Genome) khi không có lịch sử |
| **Tính cá nhân hóa** | Mỗi user nhận danh sách phim riêng biệt |

### Hướng phát triển tiếp theo:
- Tích hợp API **TMDb/IMDb** để hiển thị poster phim.
- Triển khai lên **Cloud** (Google Cloud / AWS).
- Áp dụng **Neural Collaborative Filtering** (Deep Learning).